# FPGA-evaluate — Fase A (software_now)

Valutazione pre-silicio dell'idoneita' FPGA (Zynq-7020) dei 4 champion, cross-champion.
46 figure a **dati reali** (10 sezioni) + CSV deliverable, dalle 5 librerie Fase A (`weight_profiler`, `state_profiler`, `latency_model`, `seu_inject`, `io_hil`).

Le figure 🟢 sono calcolate dai tensori/forward reali; le 🟡/🔴 (HDL/board) sono STIME di progetto marcate. Output in `results/evaluate/FPGA/`.

In [ ]:
# ENV -- champions + loader robusto + helper
import os, sys, json, glob, subprocess, importlib.util as _imu
sys.path.insert(0, os.getcwd())
for pkg in ['pandas', 'matplotlib', 'numpy', 'torch', 'scipy']:
    if _imu.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
import numpy as np, pandas as pd, torch
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import scripts.fpga_figures as FF

RESULTS = 'results/evaluate/FPGA'
BRANCH = 'EventProp_Study'
CACHE = 'data/cache_1500_launch_cut0.0_ou0.0.pt'
os.makedirs(RESULTS, exist_ok=True)
assert os.path.isfile(CACHE), 'manca cache: ' + CACHE
CACHE_DATA = torch.load(CACHE, map_location='cpu', weights_only=False)

# 4 champion (2 BPTT + 2 EventProp). L'oracolo non serve al profilo FPGA.
CHAMPIONS = [
    ('Raffaello',    'R33_C2_A1_T12_fix',      '#e34948', 'baseline'),
    ('Leonardo',     'LS3_PEAK_R0_launch_d03', '#2a78d6', 'baseline'),
    ('Donatello',    'PE_t05_gp0002',          '#4a3aa7', 'eventprop_alif_full'),
    ('Michelangelo', 'A_lr1e2_t06_r16',        '#eb6834', 'eventprop_alif_full'),
]
COLORS = {a: c for a, _, c, _ in CHAMPIONS}

def robust_load(tag, variant, device='cpu'):
    from core.network import build_model
    p = os.path.join('checkpoints', tag, 'best_model.pt')
    if not os.path.isfile(p):
        return None
    ck = torch.load(p, map_location=device, weights_only=False)
    state = ck['model_state'] if isinstance(ck, dict) and 'model_state' in ck else ck
    if 'layer_hidden.rec_U' in state:
        hidden = int(state['layer_hidden.rec_U'].shape[0]); rank = int(state['layer_hidden.rec_U'].shape[1])
    else:
        hidden, rank = 32, 8
    v = 'baseline' if 'layer_out.fc_weight' in state else \
        ('eventprop_alif_full' if 'layer_out.weight' in state else variant)
    try:
        m = build_model(variant=v, hidden_size=hidden, rank=rank, max_delay=6, bit_shift=3)
        res = m.load_state_dict(state, strict=False)
    except Exception:
        return None
    if any('layer_out' in k for k in getattr(res, 'missing_keys', [])):
        print('   [WARN] %s: readout non caricato -> scartato' % tag); return None
    m.eval(); return m

MODELS = {}
for alias, tag, _, variant in CHAMPIONS:
    m = robust_load(tag, variant); MODELS[alias] = m
    print('[%-4s] %-14s <- %s' % ('OK' if m is not None else 'FAIL', alias, tag))
AVAIL = [a for a, _, _, _ in CHAMPIONS if MODELS.get(a) is not None]
print('\nChampion disponibili:', AVAIL)

def sub(d):
    p = os.path.join(RESULTS, d); os.makedirs(p, exist_ok=True); return p
def savefig(d, name, fig):
    fig.savefig(os.path.join(sub(d), name), dpi=120, bbox_inches='tight'); plt.close(fig)
def savecsv(d, name, rows):
    pd.DataFrame(rows).to_csv(os.path.join(sub(d), name), index=False)
def done_section(d):
    return os.path.isdir(os.path.join(RESULTS, d)) and \
        len(glob.glob(os.path.join(RESULTS, d, '*.png'))) >= len(FF.SECTIONS[d])

ctx = None


In [ ]:
import traceback as _tb, os as _os
try:
    # CTX -- contesto reale (una volta) con budget Azure pieno
    if not AVAIL:
        print('[skip] nessun champion disponibile'); ctx = None
    else:
        _models = {a: MODELS[a] for a in AVAIL}
        ctx = FF.build_ctx(_models, CACHE_DATA, colors={a: COLORS[a] for a in AVAIL}, hb=FF.HB_AZURE)
        ctx['models_ref'] = _models
        print('ctx costruito per', ctx['aliases'])
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_ctx.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] ctx -> ERROR_ctx.txt'); print(_tb.format_exc())


In [ ]:
import traceback as _tb, os as _os
try:
    # sezione 00_Readiness
    D = '00_Readiness'
    if ctx is None:
        print('[skip] ctx assente', D)
    elif done_section(D):
        print('[SKIP]', D)
    else:
        _n = FF.save_section(ctx, D, savefig); print('[OK]', D, '->', _n, 'figure')
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_00_Readiness.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] 00_Readiness -> ERROR_00_Readiness.txt'); print(_tb.format_exc())


In [ ]:
import traceback as _tb, os as _os
try:
    # sezione 01_Weights_po2
    D = '01_Weights_po2'
    if ctx is None:
        print('[skip] ctx assente', D)
    elif done_section(D):
        print('[SKIP]', D)
    else:
        _n = FF.save_section(ctx, D, savefig); print('[OK]', D, '->', _n, 'figure')
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_01_Weights_po2.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] 01_Weights_po2 -> ERROR_01_Weights_po2.txt'); print(_tb.format_exc())


In [ ]:
import traceback as _tb, os as _os
try:
    # sezione 02_FixedPoint
    D = '02_FixedPoint'
    if ctx is None:
        print('[skip] ctx assente', D)
    elif done_section(D):
        print('[SKIP]', D)
    else:
        _n = FF.save_section(ctx, D, savefig); print('[OK]', D, '->', _n, 'figure')
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_02_FixedPoint.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] 02_FixedPoint -> ERROR_02_FixedPoint.txt'); print(_tb.format_exc())


In [ ]:
import traceback as _tb, os as _os
try:
    # sezione 03_Spiking
    D = '03_Spiking'
    if ctx is None:
        print('[skip] ctx assente', D)
    elif done_section(D):
        print('[SKIP]', D)
    else:
        _n = FF.save_section(ctx, D, savefig); print('[OK]', D, '->', _n, 'figure')
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_03_Spiking.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] 03_Spiking -> ERROR_03_Spiking.txt'); print(_tb.format_exc())


In [ ]:
import traceback as _tb, os as _os
try:
    # sezione 04_Energy
    D = '04_Energy'
    if ctx is None:
        print('[skip] ctx assente', D)
    elif done_section(D):
        print('[SKIP]', D)
    else:
        _n = FF.save_section(ctx, D, savefig); print('[OK]', D, '->', _n, 'figure')
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_04_Energy.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] 04_Energy -> ERROR_04_Energy.txt'); print(_tb.format_exc())


In [ ]:
import traceback as _tb, os as _os
try:
    # sezione 05_Timing_WCET
    D = '05_Timing_WCET'
    if ctx is None:
        print('[skip] ctx assente', D)
    elif done_section(D):
        print('[SKIP]', D)
    else:
        _n = FF.save_section(ctx, D, savefig); print('[OK]', D, '->', _n, 'figure')
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_05_Timing_WCET.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] 05_Timing_WCET -> ERROR_05_Timing_WCET.txt'); print(_tb.format_exc())


In [ ]:
import traceback as _tb, os as _os
try:
    # sezione 06_Resources_DSE
    D = '06_Resources_DSE'
    if ctx is None:
        print('[skip] ctx assente', D)
    elif done_section(D):
        print('[SKIP]', D)
    else:
        _n = FF.save_section(ctx, D, savefig); print('[OK]', D, '->', _n, 'figure')
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_06_Resources_DSE.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] 06_Resources_DSE -> ERROR_06_Resources_DSE.txt'); print(_tb.format_exc())


In [ ]:
import traceback as _tb, os as _os
try:
    # sezione 07_SEU_ISO26262
    D = '07_SEU_ISO26262'
    if ctx is None:
        print('[skip] ctx assente', D)
    elif done_section(D):
        print('[SKIP]', D)
    else:
        _n = FF.save_section(ctx, D, savefig); print('[OK]', D, '->', _n, 'figure')
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_07_SEU_ISO26262.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] 07_SEU_ISO26262 -> ERROR_07_SEU_ISO26262.txt'); print(_tb.format_exc())


In [ ]:
import traceback as _tb, os as _os
try:
    # sezione 08_IO_HIL
    D = '08_IO_HIL'
    if ctx is None:
        print('[skip] ctx assente', D)
    elif done_section(D):
        print('[SKIP]', D)
    else:
        _n = FF.save_section(ctx, D, savefig); print('[OK]', D, '->', _n, 'figure')
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_08_IO_HIL.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] 08_IO_HIL -> ERROR_08_IO_HIL.txt'); print(_tb.format_exc())


In [ ]:
import traceback as _tb, os as _os
try:
    # sezione 09_Thermal
    D = '09_Thermal'
    if ctx is None:
        print('[skip] ctx assente', D)
    elif done_section(D):
        print('[SKIP]', D)
    else:
        _n = FF.save_section(ctx, D, savefig); print('[OK]', D, '->', _n, 'figure')
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_09_Thermal.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] 09_Thermal -> ERROR_09_Thermal.txt'); print(_tb.format_exc())


In [ ]:
import traceback as _tb, os as _os
try:
    # CSV deliverable (§4.3)
    if ctx is not None:
        FF.save_all_csvs(ctx, savecsv); print('[OK] CSV deliverable salvati')
except Exception:
    _os.makedirs(RESULTS, exist_ok=True)
    open(_os.path.join(RESULTS, 'ERROR_csv.txt'), 'w', encoding='utf-8').write(_tb.format_exc())
    print('[ERROR] csv -> ERROR_csv.txt'); print(_tb.format_exc())


In [ ]:
# push dei risultati FPGA (robusto, retry)
import subprocess, time
def _git(*a): return subprocess.run(['git', *a], capture_output=True, text=True)
_git('add', RESULTS)
r = _git('commit', '-m', 'eval FPGA Fase A: 46 figure (10 sezioni) + CSV deliverable, cross-champion')
if r.returncode != 0 and 'nothing to commit' in (r.stdout + r.stderr):
    print('niente da committare')
else:
    for k in range(5):
        _git('pull', '--no-rebase', '--no-edit', 'origin', BRANCH)
        p = _git('push', 'origin', BRANCH)
        if p.returncode == 0:
            print('push OK'); break
        print('push retry', k, p.stderr[-160:]); time.sleep(3)
